# Module 8 – Apache Spark with Titanic Dataset

This notebook demonstrates how to use Apache Spark with the Titanic dataset using PySpark. We’ll:
- Set up Spark
- Load a dataset
- Convert it to Spark DataFrame
- Perform basic Spark operations
- Run SQL queries


## Step 1: Environment

This notebook runs on **Databricks Free Edition** — see [DATABRICKS_SETUP.md](./DATABRICKS_SETUP.md) in this folder for account setup and importing this notebook. No local PySpark, JDK, or WSL install is needed.

Databricks notebooks already provide a `spark` session in scope, so the cell below works as-is — it just reuses the existing session instead of creating a new one.

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("TitanicSparkDemo").getOrCreate()

spark

## Step 2: Load Titanic Dataset
We’ll use Seaborn’s version of Titanic dataset for simplicity.

In [2]:
import seaborn as sns
import pandas as pd

# Load and filter columns
df = sns.load_dataset("titanic")
df = df[['survived', 'pclass', 'age', 'sex', 'fare']].dropna()

df

,survived,pclass,age,sex,fare
0,0,3,22.0,male,7.2500
1,1,1,38.0,female,71.2833
2,1,3,26.0,female,7.9250
3,1,1,35.0,female,53.1000
4,0,3,35.0,male,8.0500
...,...,...,...,...,...
885,0,3,39.0,female,29.1250
886,0,2,27.0,male,13.0000
887,1,1,19.0,female,30.0000
889,1,1,26.0,male,30.0000


## Step 3: Convert to Spark DataFrame

In [3]:
df_spark = spark.createDataFrame(df)


df_spark.printSchema()
df_spark.show()

root
 |-- survived: long (nullable = true)
 |-- pclass: long (nullable = true)
 |-- age: double (nullable = true)
 |-- sex: string (nullable = true)
 |-- fare: double (nullable = true)



[Stage 0:>                                                          (0 + 1) / 1]

+--------+------+----+------+-------+
|survived|pclass| age|   sex|   fare|
+--------+------+----+------+-------+
|       0|     3|22.0|  male|   7.25|
|       1|     1|38.0|female|71.2833|
|       1|     3|26.0|female|  7.925|
|       1|     1|35.0|female|   53.1|
|       0|     3|35.0|  male|   8.05|
|       0|     1|54.0|  male|51.8625|
|       0|     3| 2.0|  male| 21.075|
|       1|     3|27.0|female|11.1333|
|       1|     2|14.0|female|30.0708|
|       1|     3| 4.0|female|   16.7|
|       1|     1|58.0|female|  26.55|
|       0|     3|20.0|  male|   8.05|
|       0|     3|39.0|  male| 31.275|
|       0|     3|14.0|female| 7.8542|
|       1|     2|55.0|female|   16.0|
|       0|     3| 2.0|  male| 29.125|
|       0|     3|31.0|female|   18.0|
|       0|     2|35.0|  male|   26.0|
|       1|     2|34.0|  male|   13.0|
|       1|     3|15.0|female| 8.0292|
+--------+------+----+------+-------+
only showing top 20 rows


## Step 4: Spark DataFrame Operations
Basic exploration and transformations.

In [4]:
# Average age of passengers
df_spark.groupBy("pclass").avg("age").show()

# Count survivors by class
df_spark.groupBy("pclass", "survived").count().show()

Py4JJavaError: An error occurred while calling o69.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 1.0 failed 1 times, most recent failure: Lost task 0.0 in stage 1.0 (TID 1) (host.docker.internal executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed). Consider setting 'spark.sql.execution.pyspark.udf.faulthandler.enabled' or'spark.python.worker.faulthandler.enabled' configuration to 'true' for the better Python traceback.
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:678)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:663)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:35)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:1034)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:1014)
	at org.

### Interpreting Spark SQL Queries on Titanic Data

In this step, we run **SQL-like queries** on the Spark DataFrame to explore the Titanic dataset:

1. **Average Age by Passenger Class**:
   - Gives insight into how passenger demographics differed across travel classes.
   - Higher-class passengers often had a higher average age.

2. **Survival Count by Class**:
   - Shows how survival was distributed between classes.
   - Useful for understanding social/economic influence on survival probability.

3. **Average Fare by Class and Gender**:
   - Highlights how ticket cost varied across classes and between male/female passengers.

These queries demonstrate how **Spark SQL** allows scalable and intuitive exploration of big data.



## Step 5: Spark SQL
We can use SQL queries by registering a temporary view.

In [5]:
df_spark.createOrReplaceTempView("titanic")


# Query 1: Average age by passenger class
avg_age_query = spark.sql("""
    SELECT Pclass, ROUND(AVG(Age), 2) AS Average_Age
    FROM titanic
    GROUP BY Pclass
    ORDER BY Pclass  
""")
print("Average Age by Passenger Class:")
avg_age_query.show()


# Query 2: Count of survivors by passenger class
survival_count_query = spark.sql("""
    SELECT Pclass, Survived, COUNT(*) AS Count
    FROM titanic
    GROUP BY Pclass, Survived
    ORDER BY Pclass, Survived
""")
print("Survival Count by Class:")
survival_count_query.show()

# # Query 3: Average Fare by class and gender
avg_fare_query = spark.sql("""
    SELECT Pclass, Sex, ROUND(AVG(Fare), 2) AS Average_Fare
    FROM titanic
    GROUP BY Pclass, Sex
    ORDER BY Pclass, Sex
""")
print("Average Fare by Class and Gender:")
avg_fare_query.show(truncate=False)

 

Average Age by Passenger Class:


Py4JJavaError: An error occurred while calling o78.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 2 in stage 2.0 failed 1 times, most recent failure: Lost task 2.0 in stage 2.0 (TID 11) (host.docker.internal executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed). Consider setting 'spark.sql.execution.pyspark.udf.faulthandler.enabled' or'spark.python.worker.faulthandler.enabled' configuration to 'true' for the better Python traceback.
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:678)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:663)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:35)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:1034)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:1014)
	at org